## Preprocess and Map Socioeconomic Vulnerability Data in Busan (2023)
This workflow preprocesses socioeconomic vulnerability data for Busan, South Korea, using the number of general basic livelihood security recipients as a vulnerability indicator.
To ensure consistency with the population data provided by the Busan Data Innovation Center, this analysis extracts recipient data for 2023 while using administrative dong boundaries and codes from June 2025, the latest available version. The recipient data is then joined to the administrative boundaries to create a spatial dataset for subsequent flood risk analysis.

### Extract Busan Administrative Boundaries from the Excel Code Table

In [1]:
# Load the latest available administrative dong codes (June 2025) and extract Busan
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

code_path = "C:/Users/nayou/OneDrive/문서/공모전/2026 부산 Big Data 활용대회/데이터 수집/BND_ADM_DONG_PG/센서스 공간정보 지역 코드.xlsx"

# Read the latest available sheet (June 2025)
code = pd.read_excel(code_path, sheet_name="2025년 6월", header=1)

# Extract administrative dongs in Busan
busan_code = code[code["시도명칭"] == "부산광역시"].copy()

print(busan_code.head())
print("부산 행정동 수:", len(busan_code))

     시도코드   시도명칭  시군구코드 시군구명칭  읍면동코드 읍면동명칭
426    21  부산광역시     10    중구    510   중앙동
427    21  부산광역시     10    중구    520   동광동
428    21  부산광역시     10    중구    530   대청동
429    21  부산광역시     10    중구    540   보수동
430    21  부산광역시     10    중구    560   부평동
부산 행정동 수: 206


In [2]:
import geopandas as gpd

# Generate the 8-digit administrative dong code (ADM_CD)
busan_code["ADM_CD"] = (
    busan_code["시도코드"].astype(int).astype(str).str.zfill(2) # Add leading zeros to match the required number of digits
    + busan_code["시군구코드"].astype(int).astype(str).str.zfill(3)
    + busan_code["읍면동코드"].astype(int).astype(str).str.zfill(3)
)

print(busan_code[["시군구명칭", "읍면동명칭", "ADM_CD"]].head()) 

    시군구명칭 읍면동명칭    ADM_CD
426    중구   중앙동  21010510
427    중구   동광동  21010520
428    중구   대청동  21010530
429    중구   보수동  21010540
430    중구   부평동  21010560


### Extract Busan Administrative Boundaries from the Shapefile and Compare with the Excel Code Table

In [3]:
# Extract Busan administrative dong boundaries using ADM_CD
shp_path = r"C:\Users\nayou\OneDrive\문서\공모전\2026 부산 Big Data 활용대회\데이터 수집\BND_ADM_DONG_PG\BND_ADM_DONG_PG.shp"

dong = gpd.read_file(shp_path)

# Convert ADM_CD to string for consistent matching
dong["ADM_CD"] = dong["ADM_CD"].astype(str)

# Select administrative dongs matching the Busan code list
busan_dong = dong[dong["ADM_CD"].isin(busan_code["ADM_CD"])].copy()

# Compare the number of administrative dongs
print("Number of administrative dongs in the code table:", len(busan_code))
print("Number of administrative dongs extracted from the shapefile:", len(busan_dong))
print(busan_dong[["ADM_CD", "ADM_NM"]].head())

Number of administrative dongs in the code table: 206
Number of administrative dongs extracted from the shapefile: 206
       ADM_CD ADM_NM
426  21010510    중앙동
427  21010520    동광동
428  21010530    대청동
429  21010540    보수동
430  21010560    부평동


### Save the extracted Busan administrative dong boundaries as a shapefile

In [5]:
out_path = r"C:\Users\nayou\OneDrive\문서\공모전\2026 부산 Big Data 활용대회\복기\Busan_ADM_DONG_2025.shp"
busan_dong.to_file(out_path, encoding="utf-8")

### Process Vulnerable Population Data

In [6]:
vulnerability_file = "C:/Users/nayou/OneDrive/문서/공모전/2026 부산 Big Data 활용대회/데이터 수집/기초수급자 현황/부산광역시 기초생활보장 수급권자구분별 총괄 현황(2021-2023)2.csv"

df_vul = pd.read_csv(vulnerability_file, encoding="cp949")

print(df_vul.columns.tolist())
print(df_vul.head())

['연도', '시도', '시군구', '읍면동', '일반수급자 가구수', '일반수급자 수급권자수', '조건부수급자 가구수', '조건부수급자 수급권자수', '특례수급자 가구수', '특례수급자 수급권자수', '기타 가구수', '기타 수급권자수', '시설수급자 가구수', '시설수급자 수급권자수']
     연도     시도 시군구  읍면동  일반수급자 가구수  일반수급자 수급권자수  조건부수급자 가구수  조건부수급자 수급권자수  \
0  2021  부산광역시  중구   중구          0            0           0             0   
1  2021  부산광역시  중구  중앙동        121          154          18            24   
2  2021  부산광역시  중구  동광동        316          362          30            44   
3  2021  부산광역시  중구  대청동        432          537          54           103   
4  2021  부산광역시  중구  보수동        710          849          94           158   

   특례수급자 가구수  특례수급자 수급권자수  기타 가구수  기타 수급권자수  시설수급자 가구수  시설수급자 수급권자수  
0          0            0       0         0          4            4  
1         11           11       0         0         12           12  
2          3            3       0         0          0            0  
3         15           19       0         0          0            0  
4         22           2

In [7]:
# Extract vulnerability data for 2023
vul_2023 = df_vul[df_vul["연도"] == 2023].copy()

# Remove district-level aggregate rows (e.g., "중구-중구")
vul_2023 = vul_2023[vul_2023["읍면동"] != vul_2023["시군구"]].copy()

# Create a column for the number of basic livelihood security recipients
vul_2023["기초수급자수"] = vul_2023["일반수급자 수급권자수"]

print(vul_2023[["시군구", "읍면동", "기초수급자수"]])

     시군구  읍면동  기초수급자수
446   중구  중앙동     153
447   중구  동광동     401
448   중구  대청동     560
449   중구  보수동     986
450   중구  부평동     451
..   ...  ...     ...
662  기장군  장안읍     552
663  기장군  정관읍    4454
664  기장군  일광읍     728
665  기장군  일광면       2
666  기장군  철마면     745

[206 rows x 3 columns]


In [8]:
# Add district and dong names to the shapefile
busan_dong2 = busan_dong.merge(busan_code[["ADM_CD", "시군구명칭", "읍면동명칭"]], on="ADM_CD", how="left")

# Join 2023 basic livelihood security recipient data
busan_vul = busan_dong2.merge(vul_2023[["시군구", "읍면동", "기초수급자수"]], left_on=["시군구명칭", "읍면동명칭"], right_on=["시군구", "읍면동"], how="left")

# Check the results
print("Total number of administrative dongs:", len(busan_vul))
print("Number of unmatched records:", busan_vul["기초수급자수"].isna().sum())

print(busan_vul.loc[busan_vul["기초수급자수"].isna(), ["ADM_CD", "시군구명칭", "읍면동명칭"]])

Total number of administrative dongs: 206
Number of unmatched records: 1
       ADM_CD 시군구명칭 읍면동명칭
163  21120562   강서구   신호동


### Investigate the unmatched administrative dong (Sinho-dong, Gangseo-gu)

In [9]:
# Check administrative dong names in Gangseo-gu for 2023
print(df_vul[(df_vul["연도"] == 2023) & (df_vul["시군구"] == "강서구")][["시군구", "읍면동"]])

     시군구   읍면동
614  강서구   강서구
615  강서구  대저1동
616  강서구  대저2동
617  강서구   강동동
618  강서구  명지1동
619  강서구  명지2동
620  강서구   가락동
621  강서구   녹산동
622  강서구  가덕도동


In [10]:
print(busan_code[busan_code["시군구명칭"] == "강서구"][["시군구명칭", "읍면동명칭", "ADM_CD"]])

    시군구명칭 읍면동명칭    ADM_CD
584   강서구  대저1동  21120510
585   강서구  대저2동  21120520
586   강서구   강동동  21120530
587   강서구   가락동  21120550
588   강서구   녹산동  21120561
589   강서구   신호동  21120562
590   강서구  가덕도동  21120580
591   강서구  명지1동  21120590
592   강서구  명지2동  21120600


### Save

In [ ]:
# Save administrative dong boundaries with vulnerability data as a GeoPackage
out_gpkg = "C:/Users/nayou/OneDrive/문서/공모전/2026 부산 Big Data 활용대회/복기/Busan_ADM_DONG_2025_vulnerability.gpkg"
busan_vul.to_file(out_gpkg, layer="vulnerability", driver="GPKG")